# dbt Advanced — Macros, Packages, Exposures, CI/CD

## Mental Model

dbt is the transformation control plane for analytics engineering.

- **Macros** keep repeated SQL logic DRY and composable.
- **Packages** extend dbt with community-tested building blocks.
- **Exposures** connect warehouse models to downstream dashboards and apps.
- **CI/CD** lets teams validate only what changed, so feedback stays fast even in large projects.

This notebook uses the live Citi telemetry learning stack and writes runnable dbt assets into the `citi_dbt` project.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import textwrap
from pathlib import Path

import psycopg2
from psycopg2.extras import RealDictCursor
from IPython.display import Markdown, display

WORKSPACE_DIR = Path(r"D:\Workspace\Technologies")
PROJECT_DIR = WORKSPACE_DIR / "citi_dbt"
DBT_EXE = Path(r"C:\py_venv\proj_educate\Scripts\dbt.exe")
PROFILES_DIR = Path.home() / ".dbt"
TARGET_NAME = "postgres"

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!",
}

print(f"Python platform: {platform.platform()}")
print(f"Workspace dir: {WORKSPACE_DIR}")
print(f"Project dir:   {PROJECT_DIR}")
print(f"dbt exe:       {DBT_EXE}")
print(f"Profiles dir:  {PROFILES_DIR}")

assert PROJECT_DIR.exists(), f"Expected dbt project directory does not exist: {PROJECT_DIR}"
assert DBT_EXE.exists(), f"Expected dbt executable does not exist: {DBT_EXE}"
assert PROFILES_DIR.exists(), f"Expected dbt profiles directory does not exist: {PROFILES_DIR}"

for required in ["models", "macros"]:
    path = PROJECT_DIR / required
    path.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, cwd=PROJECT_DIR, check=True):
    print("\n" + "=" * 100)
    print("RUN:", " ".join(str(x) for x in cmd))
    print("CWD:", cwd)
    print("=" * 100)
    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        capture_output=True,
        shell=False,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(str(x) for x in cmd)}")
    return result

def run_dbt(args, check=True):
    cmd = [DBT_EXE, *args, "--project-dir", PROJECT_DIR, "--profiles-dir", PROFILES_DIR, "--target", TARGET_NAME]
    return run_cmd(cmd, cwd=PROJECT_DIR, check=check)

def write_text(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip("\n"), encoding="utf-8")
    print(f"Wrote {path}")

def pg_query(sql: str):
    with psycopg2.connect(**DB_CONFIG) as conn:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute(sql)
            rows = cur.fetchall()
    return rows

source_counts = pg_query("""
select 'endpoints' as table_name, count(*) as row_count from endpoints
union all
select 'metrics' as table_name, count(*) as row_count from metrics
union all
select 'alerts' as table_name, count(*) as row_count from alerts
order by table_name;
""")
print("Source table counts:")
for row in source_counts:
    print(row)

## 1) Imports + dbt setup

The next cells create runnable dbt assets directly inside the `citi_dbt` project.

In [ ]:
macro_sql = """
{% macro cents_to_dollars(column_name) -%}
    (({{ column_name }}::numeric) / 100.0)
{%- endmacro %}

{% macro current_timestamp_utc() -%}
    (timezone('UTC', now()))
{%- endmacro %}

{% macro generate_surrogate_key(columns) -%}
    md5(
        concat_ws(
            '||',
            {% for col in columns %}
            coalesce(cast({{ col }} as text), '_dbt_null_'){% if not loop.last %}, {% endif %}
            {% endfor %}
        )
    )
{%- endmacro %}
"""
write_text(PROJECT_DIR / "macros" / "citi_helpers.sql", macro_sql)

macro_model_sql = """
{{ config(materialized='view') }}

with alert_base as (
    select
        a.alert_id,
        a.endpoint_id,
        a.severity,
        a.message,
        a.created_at,
        e.name as endpoint_name,
        e.region,
        e.status,
        e.category
    from alerts a
    join endpoints e
      on a.endpoint_id = e.endpoint_id
),
final as (
    select
        {{ generate_surrogate_key(["alert_id", "endpoint_id", "severity", "created_at"]) }} as alert_sk,
        alert_id,
        endpoint_id,
        endpoint_name,
        region,
        status,
        category,
        severity,
        message,
        created_at,
        {{ current_timestamp_utc() }} as transformed_at_utc,
        {{ cents_to_dollars("length(message) * 100") }} as message_length_dollars_demo
    from alert_base
)
select *
from final
"""
write_text(PROJECT_DIR / "models" / "mart_alert_macro_demo.sql", macro_model_sql)

alert_summary_sql = """
{{ config(materialized='table') }}

with base as (
    select
        a.severity,
        e.region,
        e.category,
        count(*) as alert_count,
        avg(length(a.message))::numeric(12,2) as avg_message_length
    from alerts a
    join endpoints e
      on a.endpoint_id = e.endpoint_id
    group by 1, 2, 3
)
select *
from base
"""
write_text(PROJECT_DIR / "models" / "mart_alert_summary.sql", alert_summary_sql)

schema_yml = """
version: 2

models:
  - name: mart_alert_macro_demo
    description: "Demonstrates custom Citi macros against the live telemetry dataset."
    columns:
      - name: alert_sk
        tests:
          - not_null
          - unique

  - name: mart_alert_summary
    description: "Aggregated alert summary by severity, region, and category."
    tests:
      - assert_severity_distribution:
          max_ratio: 0.40

exposures:
  - name: citi_telemetry_dashboard
    type: dashboard
    maturity: high
    url: https://citi.example.internal/dashboards/telemetry
    description: >
      Citi Telemetry Dashboard for monitored API endpoints, alert volume, and severity trends.
    depends_on:
      - ref('mart_alert_summary')
    owner:
      name: Sean Girgis
      email: sean.girgis@example.com
"""
write_text(PROJECT_DIR / "models" / "schema.yml", schema_yml)

generic_test_sql = """
{% test assert_severity_distribution(model, max_ratio=0.40) %}

with severity_counts as (
    select
        severity,
        sum(alert_count) as severity_count
    from {{ model }}
    group by 1
),
totals as (
    select sum(severity_count) as total_count
    from severity_counts
),
violations as (
    select
        s.severity,
        s.severity_count,
        t.total_count,
        (s.severity_count::numeric / nullif(t.total_count, 0)) as severity_ratio
    from severity_counts s
    cross join totals t
    where (s.severity_count::numeric / nullif(t.total_count, 0)) > {{ max_ratio }}
)
select *
from violations

{% endtest %}
"""
write_text(PROJECT_DIR / "tests" / "generic" / "assert_severity_distribution.sql", generic_test_sql)

## 2) Macros

Create `macros/citi_helpers.sql`, use those macros in a model, run dbt, then verify output.

In [ ]:
run_dbt(["debug"])
run_dbt(["run", "--select", "mart_alert_macro_demo", "mart_alert_summary"])

macro_verification = pg_query("""
select
    alert_sk,
    alert_id,
    endpoint_id,
    severity,
    endpoint_name,
    region,
    transformed_at_utc,
    message_length_dollars_demo
from mart_alert_macro_demo
order by created_at desc
limit 5;
""")

print("Macro verification sample:")
for row in macro_verification:
    print(row)

## 3) Packages

Add `dbt_utils` to `packages.yml`, run `dbt deps`, use it in a model, and verify it runs.

In [ ]:
packages_yml = """
packages:
  - package: dbt-labs/dbt_utils
    version: [">=1.1.1", "<2.0.0"]
"""
write_text(PROJECT_DIR / "packages.yml", packages_yml)

dbt_utils_model_sql = """
{{ config(materialized='view') }}

select
    {{ dbt_utils.generate_surrogate_key(["cast(endpoint_id as text)", "metric_name", "cast(timestamp as text)"]) }} as metric_sk,
    endpoint_id,
    metric_name,
    value,
    timestamp
from {{ source('public', 'metrics') if false else ref('stg_metrics_placeholder') }}
"""
# The line above uses an unreachable branch to preserve SQL lint friendliness in this notebook-generated file.

# Overwrite with a runnable version that does not require pre-existing sources or staging.
dbt_utils_model_sql = """
{{ config(materialized='view') }}

select
    {{ dbt_utils.generate_surrogate_key(["'metric'", "cast(endpoint_id as text)", "metric_name", "cast(timestamp as text)"]) }} as metric_sk,
    endpoint_id,
    metric_name,
    value,
    timestamp
from metrics
"""
write_text(PROJECT_DIR / "models" / "mart_metric_surrogate_demo.sql", dbt_utils_model_sql)

run_dbt(["deps"])
run_dbt(["run", "--select", "mart_metric_surrogate_demo"])

pkg_verify = pg_query("""
select metric_sk, endpoint_id, metric_name, value, timestamp
from mart_metric_surrogate_demo
order by timestamp desc
limit 5;
""")
for row in pkg_verify:
    print(row)

print("Package dbt_utils installed and used")

## 4) Exposures

Add an exposure for the **Citi Telemetry Dashboard**, then list and inspect exposure metadata.

In [ ]:
ls_result = run_dbt(["ls", "--select", "+exposures"])
print("Exposure selection output above.")

schema_text = (PROJECT_DIR / "models" / "schema.yml").read_text(encoding="utf-8")
print("\nExposure metadata snippet from schema.yml:\n")
start_idx = schema_text.index("exposures:")
print(schema_text[start_idx:])

## 5) Custom Generic Test

Create `assert_severity_distribution` so no single severity exceeds 40% of total alert volume in the summary model, then run `dbt test`.

In [ ]:
test_result = run_dbt(["test", "--select", "mart_alert_summary"], check=False)
print(f"dbt test return code: {test_result.returncode}")

severity_dist = pg_query("""
with severity_counts as (
    select severity, count(*) as severity_count
    from alerts
    group by severity
),
totals as (
    select sum(severity_count) as total_count from severity_counts
)
select
    s.severity,
    s.severity_count,
    round((s.severity_count::numeric / nullif(t.total_count, 0))::numeric, 4) as severity_ratio
from severity_counts s
cross join totals t
order by severity_ratio desc;
""")
print("Observed source severity distribution:")
for row in severity_dist:
    print(row)

## 6) CI/CD Pattern

This is the minimal fast-feedback dbt CI flow for a large project.

In [ ]:
ci_script = r"""#!/usr/bin/env bash
set -euo pipefail

PROJECT_DIR="D:/Workspace/Technologies/citi_dbt"
PROFILES_DIR="$HOME/.dbt"
DBT="C:/py_venv/proj_educate/Scripts/dbt.exe"

"$DBT" deps --project-dir "$PROJECT_DIR" --profiles-dir "$PROFILES_DIR" --target postgres
"$DBT" compile --project-dir "$PROJECT_DIR" --profiles-dir "$PROFILES_DIR" --target postgres
"$DBT" test --select state:modified+ --project-dir "$PROJECT_DIR" --profiles-dir "$PROFILES_DIR" --target postgres
"$DBT" run --select state:modified+ --project-dir "$PROJECT_DIR" --profiles-dir "$PROFILES_DIR" --target postgres
"""
print(ci_script)

print(textwrap.dedent("""
state:modified selects nodes whose code or config changed relative to the comparison manifest.
The trailing + expands selection to related downstream or parent nodes as needed.
That means CI validates only the blast radius of a change instead of rerunning the whole project.
""").strip())

## 7) What Just Happened

Macros are dbt's DRY mechanism. Packages extend the standard library. Exposures create lineage from dbt to BI tools. CI/CD with `state:modified+` runs only what changed — the pattern that keeps feedback fast even when a dbt project grows toward hundreds of models.